# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

~~~
**One row of my result is one:** _line item in an order._
~~~

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd

# Q1: Flatten the orders to one row per line item
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4, f"Expected 4 rows, got {len(lines)}"
assert lines['qty'].sum() == 7, f"Expected total quantity to be 7, got {lines['qty'].sum()}"

display(lines)

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
# Q2: Join the vendor information on (Left Join)
vendors_df = pd.DataFrame(vendors_json)
joined = lines.merge(vendors_df, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendor_id = unmatched['vendor_id'].unique()[0]
unmatched_qty_sum = unmatched['qty'].sum()

print(f"The unmatched vendor is: {unmatched_vendor_id}")
print(f"Number of units attached to the unmatched vendor: {unmatched_qty_sum}")
display(joined)

The unmatched vendor is: V-07
Number of units attached to the unmatched vendor: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
# Q3: Total quantity per vendor name
#groupby drops NaN keys unless we specify dropna=False.
total_qty_by_vendor = joined.groupby('name', dropna=False)['qty'].sum().reset_index()

# the unmatched vendor has no matching vendor name in the vendor list, so its name appears as NaN. to ensure we don't lose its quantity in our reporting, we group with dropna=False.
display(total_qty_by_vendor)

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,NaN,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
# Q4: Which zone sold the most items?
zone_sales = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()

print("Sales by Zone:")
display(zone_sales)

Sales by Zone:


,zone,qty
0,A,4
1,NaN,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

processed_ragged = []
for order in ragged_json:
    order_copy = order.copy()
    if 'lines' not in order_copy or not order_copy['lines']:

        order_copy['lines'] = [{'item': None, 'qty': None}]
    processed_ragged.append(order_copy)


ragged_df = pd.json_normalize(processed_ragged, record_path='lines', meta=['order', 'vendor_id'])


display(ragged_df)

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,None,NaN,5,V-10
2,Fries,NaN,6,V-01


### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**The reason why it makes sense to keep things at one row per line item is that makes it very simple to look an individual product performance directly. If things were maintained at an order level, finding out item-level information, like which items are ordered frequently together, becomes more tricky than it needs to be.**

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**The practical difference is whether or not information on a transaction was properly recorded. 0 is a proper record that something wasn't ordered, while NaN means that the record is incomplete. Downstream calculations of things like supply chain forecasts would be highly inaccurate if it treated "we don't have a record of person X buying this" as the same as "we have a record of person X not buying this. A failure on the record-keeping side could compound into supply shortages based on this false assumption.**

_your answer here_